# Amazon ML Challenge — Resumable high-recall training\nThis notebook runs the repository pipeline with persistent checkpoints in Google Drive.\n\n**Important:** upload the complete `dataset/` folder to the Drive location configured below. Re-run with the **same RUN_ID** after a Colab restart; the runner resumes from the latest checkpoint. Use a new RUN_ID only when you intentionally want a fresh experiment.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nDRIVE_ROOT = '/content/drive/MyDrive/AmazonMLChallenge'\nDRIVE_DATASET = DRIVE_ROOT + '/dataset'\nDRIVE_OUTPUT = DRIVE_ROOT + '/output'\nRUN_ID = 'experiment_06'\nprint('Drive dataset:', DRIVE_DATASET)\nprint('Persistent output/checkpoints:', DRIVE_OUTPUT)\nprint('RUN_ID:', RUN_ID)

In [ ]:
import os, subprocess, shutil, pathlib\nprint('GPU check:')\nsubprocess.run(['nvidia-smi'], check=False)\nprint('\nCPU/RAM:')\nprint(open('/proc/meminfo').read().split('MemTotal:')[1].splitlines()[0])\n\nrequired = [\n    'train/train_source1.tsv', 'train/train_source2.tsv',\n    'train/train_source3.tsv', 'train/train_ground_truth.tsv',\n       'test/test_source1.tsv', 'test/test_source2.tsv', 'test/test_source3.tsv'\n]\nmissing = [p for p in required if not os.path.exists(os.path.join(DRIVE_DATASET, p))]\nprint('Missing dataset files:', missing)\nassert not missing, 'Upload the complete dataset/train and dataset/test folders to Drive first.'

In [ ]:
# Keep the large TSV reads on the Colab VM for speed; keep checkpoints/indexes on Drive.\nLOCAL_DATASET = '/content/dataset'\nos.makedirs(LOCAL_DATASET, exist_ok=True)\nprint('Copying/synchronizing dataset to local VM...')\nsubprocess.run(['rsync','-a','--info=progress2', DRIVE_DATASET + '/', LOCAL_DATASET + '/'], check=True)\nprint('Local dataset ready:', LOCAL_DATASET)

In [ ]:
REPO = '/content/Amazon-ML-Challenge'\nif os.path.exists(REPO):\n    shutil.rmtree(REPO)\nsubprocess.run(['git','clone','--depth','1','https://github.com/Hrsht02/Amazon-ML-Challenge.git',REPO],check=True)\nos.chdir(REPO + '/code/business_entity_resolution')\nsubprocess.run(['pip','install','-q','-r','requirements.txt'],check=True)\nprint('Repository and dependencies ready.')\nsubprocess.run(['git','log','--oneline','-3'],check=False)

In [ ]:
# MAIN RUN. If Colab restarts, mount Drive, repeat setup, and run this SAME cell with the SAME RUN_ID.\ncmd = [\n    'python','-u','-m','src.generate_submission',\n    '--stage','all',\n    '--run-id',RUN_ID,\n    '--dataset-dir',LOCAL_DATASET,\n    '--output-dir',DRIVE_OUTPUT,\n   ]\nprint('STARTING:', ' '.join(cmd))\nprint('All progress, recall, OOF metrics, validation metrics and resource telemetry are printed below.')\nsubprocess.run(cmd, check=True)

In [ ]:
# Resume/status inspection after a restart or if you want to inspect the saved run.\nfrom pathlib import Path\nimport json\nrd = Path(DRIVE_OUTPUT) / 'run_history' / RUN_ID\nprint('Run directory:', rd)\nfor p in sorted(rd.glob('.done_*.json')):\n    print('DONE:', p.name, json.loads(p.read_text()))\nfor name in ['audit.json','blocking_train_metrics.json','validation_metrics.json','failure.json']:\n    p=rd/name\n    if p.exists():\n        print('\n###',name)\n        print(p.read_text()[:20000])\nevents=rd/'events.jsonl'\nif events.exists():\n    print('\n### LAST EVENTS')\n    print('\n'.join(events.read_text().splitlines()[-30:]))

In [ ]:
# Final required files. These are the ONLY two files to submit.\nmatching = Path(DRIVE_OUTPUT) / 'matching_results.tsv'\ncandidates = Path(DRIVE_OUTPUT) / 'candidate_pairs.tsv'\nprint('matching_results.tsv:', matching, matching.exists(), matching.stat().st_size if matching.exists() else None)\nprint('candidate_pairs.tsv:', candidates, candidates.exists(), candidates.stat().st_size if candidates.exists() else None)\nif matching.exists():\n    print('\nMATCHING HEADER + FIRST 5 ROWS')\n    print(''.join(matching.open(encoding='utf-8').readlines()[:6]))\nif candidates.exists():\n    print('CANDIDATE HEADER + FIRST 5 ROWS')\n    print(''.join(candidates.open(encoding='utf-8').readlines()[:6]))